In [26]:
import keras
import numpy as np
import pandas as pd
from hyperopt import hp, fmin,STATUS_OK, Trials, tpe
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import mlflow
from mlflow.models import infer_signature



In [27]:
import sys
print(sys.executable)


/Users/ramrekhayadav/mlops/python/dlmlflow/.venv/bin/python


In [28]:
# load the dataset

data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [29]:
# split the dataset into traing,validation and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train  # it show all the data
test   # it show 1/4 data


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4656,6.00,0.29,0.41,10.80,0.048,55.0,149.0,0.99370,3.09,0.59,10.966667,7
3659,5.40,0.53,0.16,2.70,0.036,34.0,128.0,0.98856,3.20,0.53,13.200000,8
907,7.10,0.25,0.39,2.10,0.036,30.0,124.0,0.99080,3.28,0.43,12.200000,8
4352,7.30,0.28,0.35,1.60,0.054,31.0,148.0,0.99178,3.18,0.47,10.700000,5
3271,6.50,0.32,0.34,5.70,0.044,27.0,91.0,0.99184,3.28,0.60,12.000000,7
...,...,...,...,...,...,...,...,...,...,...,...,...
2614,6.15,0.21,0.37,3.20,0.021,20.0,80.0,0.99076,3.39,0.47,12.000000,5
755,7.10,0.28,0.44,1.80,0.032,32.0,107.0,0.99070,3.25,0.48,12.200000,7
518,5.90,0.13,0.28,1.90,0.050,20.0,78.0,0.99180,3.43,0.64,10.800000,6
3671,6.80,0.30,0.29,6.20,0.025,29.0,95.0,0.99071,3.03,0.32,12.900000,7


In [30]:
# quality is dependent variable
#rest all are independt mean training variables
X_train = train.drop(["quality"], axis=1).values
X_test=test.drop(["quality"], axis=1).values
X_train

array([[ 6.3 ,  0.25,  0.22, ...,  3.16,  0.5 , 10.5 ],
       [ 7.8 ,  0.3 ,  0.29, ...,  3.16,  0.38,  9.  ],
       [ 7.4 ,  0.38,  0.27, ...,  3.17,  0.43, 10.  ],
       ...,
       [ 7.6 ,  0.27,  0.52, ...,  3.02,  0.53, 11.4 ],
       [ 6.3 ,  0.24,  0.29, ...,  3.17,  0.38, 10.6 ],
       [ 8.1 ,  0.27,  0.35, ...,  3.22,  0.63, 10.4 ]])

In [31]:
y_train = train[["quality"]].values.ravel() # ravel() convert 2d to 1d array
y_test = test[["quality"]].values.ravel()
y_train

array([6, 6, 5, ..., 6, 6, 8])

In [32]:
# further split training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
signature= infer_signature(X_train, y_train) 

In [33]:
np.mean(X_train, axis=0)  # mean of each column

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [34]:
np.var(X_train, axis=0)  # variance of each column

array([7.27006361e-01, 1.04782780e-02, 1.41982590e-02, 2.63994276e+01,
       4.46194261e-04, 2.83726450e+02, 1.74920842e+03, 9.23159572e-06,
       2.25891723e-02, 1.24958983e-02, 1.49906782e+00])

In [35]:
X_train.shape[1]

11

In [45]:
# Ann model
import mlflow.tensorflow


def train_model(params,X_train,y_train,epochs,X_val,y_val,X_test,y_test):
    mean=np.mean(X_train, axis=0)
    var=np.var(X_train, axis=0)
    model = keras.Sequential(
        [
            keras.Input([X_train.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(
                64, activation="relu"
            ),
            keras.layers.Dense(1)
        ]
    )
    #compile the model
    model.compile(optimizer= keras.optimizers.SGD(
        learning_rate=params['lr'], momentum=params['momentum']),
        loss="mean_squared_error",
        metrics=[keras.metrics.MeanSquaredError()] 
    )
    # train the Ann model
    with mlflow.start_run(nested=True):
        model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=32)
        
        # evaluate the model on test data
        eval_results = model.evaluate(X_val, y_val, batch_size=64)
        eval_rmse=eval_results[1]
        
        #log parameters and metrics
        mlflow.log_params(params)
        mlflow.log_metric("val_rmse", eval_rmse)
        
        #log the model
        mlflow.tensorflow.log_model(model, "model", signature=signature)
        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [46]:
def objective(params):
    result=train_model(
        params,
        X_train,
        y_train,
        epochs=3,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test
    )
    
    return result

In [47]:
space={
    'lr': hp.loguniform('lr', np.log(1e-5), np.log(0.1)),
    'momentum': hp.uniform('momentum', 0.0, 1.0)
    
}

In [48]:
import mlflow.tensorflow


mlflow.set_experiment("Hyperopt_ANN_Wine_Quality")
with mlflow.start_run():
    trails=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=5,
        trials=trails
    )
    
    # fetch the details of the best run or model
    best_run=sorted(trails.results, key=lambda x: x['loss'])[0]
    
    # log the best hyperparameters
    mlflow.log_params(best)
    mlflow.log_metric("best_val_rmse", best_run['loss'])
    mlflow.tensorflow.log_model(best_run['model'], "model", signature=signature)
    print("Best hyperparameters:", best)
    print("Best validation RMSE:", best_run['loss'])
    

  0%|          | 0/5 [00:00<?, ?trial/s, best loss=?]

Epoch 1/3                                            

92/92 [==============================] - 0s 2ms/step - loss: 13.2547 - mean_squared_error: 13.2547 - val_loss: 2.9350 - val_mean_squared_error: 2.9350

Epoch 2/3                                            

92/92 [==============================] - 0s 632us/step - loss: 2.2867 - mean_squared_error: 2.2867 - val_loss: 1.9531 - val_mean_squared_error: 1.9531

Epoch 3/3                                            

92/92 [==============================] - 0s 623us/step - loss: 1.7652 - mean_squared_error: 1.7652 - val_loss: 1.6995 - val_mean_squared_error: 1.6995

12/12 [==============================] - 0s 592us/step - loss: 1.6995 - mean_squared_error: 1.6995

  0%|          | 0/5 [00:00<?, ?trial/s, best loss=?]INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpxquii9uw/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpxquii9uw/model/data/model/assets



 20%|██        | 1/5 [00:07<00:30,  7.61s/trial, best loss: 1.69947350025177]

Epoch 1/3                                                                    

92/92 [==============================] - 0s 1ms/step - loss: 34.4137 - mean_squared_error: 34.4137 - val_loss: 31.9843 - val_mean_squared_error: 31.9843

Epoch 2/3                                                                    

92/92 [==============================] - 0s 634us/step - loss: 29.7629 - mean_squared_error: 29.7629 - val_loss: 27.6742 - val_mean_squared_error: 27.6742

Epoch 3/3                                                                    

92/92 [==============================] - 0s 679us/step - loss: 25.7550 - mean_squared_error: 25.7550 - val_loss: 23.9373 - val_mean_squared_error: 23.9373

12/12 [==============================] - 0s 619us/step - loss: 23.9373 - mean_squared_error: 23.9373

 20%|██        | 1/5 [00:08<00:30,  7.61s/trial, best loss: 1.69947350025177]INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmp8n6yuv_s/model/data/model/asset

...

Epoch 1/3                                                                      

92/92 [==============================] - 0s 1ms/step - loss: 1.6676 - mean_squared_error: 1.6676 - val_loss: 0.6402 - val_mean_squared_error: 0.6402

Epoch 2/3                                                                      

92/92 [==============================] - 0s 607us/step - loss: 0.5990 - mean_squared_error: 0.5990 - val_loss: 0.5556 - val_mean_squared_error: 0.5556

Epoch 3/3                                                                      

92/92 [==============================] - 0s 662us/step - loss: 0.5626 - mean_squared_error: 0.5626 - val_loss: 0.5054 - val_mean_squared_error: 0.5054

12/12 [==============================] - 0s 584us/step - loss: 0.5054 - mean_squared_error: 0.5054

 80%|████████  | 4/5 [00:24<00:05,  5.82s/trial, best loss: 0.5643612146377563]INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpnxogxgcw/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpnxogxgcw/model/data/model/assets



100%|██████████| 5/5 [00:29<00:00,  5.96s/trial, best loss: 0.5053788423538208]
INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpszi8ekkm/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/k6/jx95ypzx09jb1dr13vzmb1180000gn/T/tmpszi8ekkm/model/data/model/assets



Best hyperparameters: {'lr': 0.026187415858248565, 'momentum': 0.698951545601299}
Best validation RMSE: 0.5053788423538208
